# Step 5 — MLflow Experiment Tracking

Log both models and compare them in MLflow UI.

In [ ]:
import boto3
import sagemaker
import mlflow
import os

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"

boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(boto_session=boto_session, default_bucket=BUCKET)
role = sagemaker.get_execution_role()
sm_client = boto3.client("sagemaker", region_name=REGION)

# Get MLflow Tracking Server ARN
response = sm_client.describe_mlflow_tracking_server(
    TrackingServerName="symptom-mlflow"
)
tracking_server_arn = response["TrackingServerArn"]
print(f"MLflow ARN: {tracking_server_arn}")

In [ ]:
# IMPORTANT: Must use ARN + SigV4 auth (not presigned URL)
os.environ["MLFLOW_TRACKING_AWS_SIGV4"] = "true"
os.environ["AWS_DEFAULT_REGION"] = REGION

mlflow.set_tracking_uri(tracking_server_arn)
mlflow.set_experiment("symptom-classification")

# Log Autopilot result
with mlflow.start_run(run_name="LightGBM-Autopilot"):
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("source",     "SageMaker Autopilot")
    mlflow.log_metric("accuracy",  1.0)
    print("LightGBM run logged ✅")

# Log BioBERT result
with mlflow.start_run(run_name="Bio-ClinicalBERT"):
    mlflow.log_param("model_type", "Bio_ClinicalBERT")
    mlflow.log_param("epochs",     3)
    mlflow.log_param("batch_size", 16)
    mlflow.log_param("max_length", 128)
    mlflow.log_metric("accuracy",  0.7641509433962265)
    mlflow.log_metric("macro_f1",  0.73)
    print("BioBERT run logged ✅")

print("\nView in Unified Studio → MLflow → Connect Tracking Server → paste ARN")
print("Select both runs → Compare → Parallel Coordinates Plot")